# Extract Semantic Models using semantic-link-labs
Extract granular metadata from Semantic Models including:
- Tables
- Columns (with data types, formats)
- Measures (with DAX expressions)
- Relationships
- Calculated columns and functions

Uses **semantic-link-labs** for comprehensive metadata extraction.

## 1. Imports

**Note**: Packages loaded from Fabric Environment (no installation needed).

In [ ]:
# All packages available from attached Fabric Environment
from sempy_labs.tom import connect_semantic_model
import sempy.fabric as fabric
from sempy.fabric import list_datasets
import json
import requests
from datetime import datetime
from notebookutils import mssparkutils

print("✅ Imports successful (from environment)")

## 2. Configuration
Parameters passed from custom item via Spark Livy API.

In [ ]:
# These will be provided as parameters when triggered via Livy
# For testing, set manually:
WORKSPACE_IDS = ["your-workspace-id"]  # Can extract from multiple workspaces
LAKEHOUSE_ID = "your-lakehouse-id"

print(f"Extracting from {len(WORKSPACE_IDS)} workspace(s)")

## 3. Extract Semantic Models

In [ ]:
def extract_semantic_model(model_id: str, workspace_id: str) -> dict:
    """Extract comprehensive metadata from a Semantic Model"""
    print(f"\n📊 Extracting Semantic Model: {model_id}")
    
    extraction_start = datetime.now()
    
    try:
        # Extract all metadata using TOMWrapper from semantic-link-labs
        print("  └─ Connecting to semantic model...")
        with connect_semantic_model(dataset=model_id, workspace=workspace_id, readonly=True) as tom:
            
            print("  └─ Extracting measures...")
            measures = []
            for measure in tom.all_measures():
                measures.append({
                    "Name": measure.Name,
                    "Expression": measure.Expression,
                    "Description": measure.Description if hasattr(measure, 'Description') else None,
                    "Table": measure.Parent.Name,
                    "FormatString": measure.FormatString if hasattr(measure, 'FormatString') else None
                })
            
            print("  └─ Extracting columns...")
            columns = []
            for column in tom.all_columns():
                columns.append({
                    "Name": column.Name,
                    "DataType": str(column.DataType),
                    "Description": column.Description if hasattr(column, 'Description') else None,
                    "Table": column.Parent.Name,
                    "SourceColumn": column.SourceColumn if hasattr(column, 'SourceColumn') else None
                })
            
            print("  └─ Extracting relationships...")
            relationships = []
            for rel in tom.model.Model.Relationships:
                relationships.append({
                    "Name": rel.Name if hasattr(rel, 'Name') else f"{rel.FromTable.Name}_{rel.ToTable.Name}",
                    "FromTable": rel.FromTable.Name,
                    "FromColumn": rel.FromColumn.Name,
                    "ToTable": rel.ToTable.Name,
                    "ToColumn": rel.ToColumn.Name,
                    "CrossFilteringBehavior": str(rel.CrossFilteringBehavior)
                })
            
            print("  └─ Extracting functions...")
            functions = []
            for function in tom.all_functions():
                functions.append({
                    "Name": function.Name,
                    "Expression": function.Expression,
                    "Description": function.Description if hasattr(function, 'Description') else None
                })
        
        extraction_end = datetime.now()
        duration = (extraction_end - extraction_start).total_seconds()
        
        # Build result structure
        result = {
            "artifactId": model_id,
            "artifactType": "SemanticModel",
            "workspaceId": workspace_id,
            "timestamp": extraction_end.isoformat(),
            "data": {
                "measures": measures,
                "columns": columns,
                "relationships": relationships,
                "functions": functions,
                "counts": {
                    "measures": len(measures),
                    "columns": len(columns),
                    "relationships": len(relationships),
                    "functions": len(functions)
                }
            },
            "metadata": {
                "extractionDuration": duration,
                "status": "success"
            }
        }
        
        print(f"  ✅ Extracted: {len(measures)} measures, {len(columns)} columns, {len(relationships)} relationships")
        return result
        
    except Exception as e:
        print(f"  ❌ Error: {str(e)}")
        return {
            "artifactId": model_id,
            "artifactType": "SemanticModel",
            "workspaceId": workspace_id,
            "timestamp": datetime.now().isoformat(),
            "data": {},
            "metadata": {
                "status": "error",
                "errorMessage": str(e)
            }
        }

print("✅ Extraction function defined")

## 4. Process All Workspaces

In [ ]:
# Get all Semantic Models from target workspaces
all_results = []

for workspace_id in WORKSPACE_IDS:
    print(f"\n🔍 Processing workspace: {workspace_id}")
    
    try:
        # Get list of Semantic Models in workspace using fabric library
        datasets_df = list_datasets(workspace=workspace_id)
        
        if datasets_df.empty:
            print(f"  No Semantic Models found in workspace")
            continue
            
        models = datasets_df.to_dict('records')
        print(f"  Found {len(models)} Semantic Model(s)")
        
        # Extract each model
        for model in models:
            # Use 'Dataset ID' column name from list_datasets result
            model_id = model.get("Dataset ID") or model.get("id")
            model_name = model.get("Dataset Name") or model.get("name") or model_id
            
            print(f"  Processing: {model_name}")
            result = extract_semantic_model(model_id, workspace_id)
            all_results.append(result)
            
            # Save individual result to lakehouse (use relative paths)
            file_path = f"Files/lineage/raw/semantic_models/{model_id}.json"
            
            # Ensure directory exists (relative path from workspace root)
            mssparkutils.fs.mkdirs("Files/lineage/raw/semantic_models")
            
            # Write JSON
            json_str = json.dumps(result, indent=2)
            mssparkutils.fs.put(file_path, json_str, overwrite=True)
            print(f"    └─ Saved: {file_path}")
            
    except Exception as e:
        print(f"  ❌ Workspace error: {str(e)}")

print(f"\n✅ Extraction complete: {len(all_results)} Semantic Model(s) processed")

## 5. Summary

In [ ]:
# Display summary
success_count = sum(1 for r in all_results if r["metadata"]["status"] == "success")
error_count = len(all_results) - success_count

print("\n" + "="*50)
print("EXTRACTION SUMMARY")
print("="*50)
print(f"Total Semantic Models: {len(all_results)}")
print(f"✅ Successful: {success_count}")
print(f"❌ Errors: {error_count}")
print("\nResults saved to: Files/lineage/raw/semantic_models/")
print("="*50)

---
# 🧪 Testing Guide

## How to Test Semantic Model Extraction

### Option 1: Test Single Model (Recommended)
Use the test cell below to extract ONE model before running full workspace extraction.

### Option 2: Test Full Extraction
Update configuration (cell 4) and run all cells for complete workspace extraction.

## 🧪 Test: Extract Single Model

In [ ]:
# TEST: Extract a single Semantic Model
# Replace these with actual values from your workspace

TEST_WORKSPACE_ID = "your-workspace-id"  # Get from workspace URL
TEST_MODEL_ID = "your-model-id"          # Get from Semantic Model properties

if TEST_WORKSPACE_ID != "your-workspace-id" and TEST_MODEL_ID != "your-model-id":
    print("🧪 Testing single model extraction...\n")
    
    # Extract the model
    test_result = extract_semantic_model(TEST_MODEL_ID, TEST_WORKSPACE_ID)
    
    # Display results
    if test_result["metadata"]["status"] == "success":
        counts = test_result["data"]["counts"]
        print(f"\n✅ SUCCESS!")
        print(f"   Measures: {counts['measures']}")
        print(f"   Columns: {counts['columns']}")
        print(f"   Relationships: {counts['relationships']}")
        print(f"   Functions: {counts['functions']}")
        print(f"   Duration: {test_result['metadata']['extractionDuration']:.2f}s")
        
        # Show sample measures
        if counts['measures'] > 0:
            print(f"\n📊 Sample Measures:")
            for i, measure in enumerate(test_result["data"]["measures"][:3]):
                print(f"   {i+1}. {measure.get('Measure Name', 'N/A')}")
                if 'Measure Expression' in measure:
                    expr = measure['Measure Expression'][:100]
                    print(f"      Expression: {expr}...")
        
        # Show sample columns
        if counts['columns'] > 0:
            print(f"\n📋 Sample Columns:")
            for i, column in enumerate(test_result["data"]["columns"][:3]):
                print(f"   {i+1}. {column.get('Column Name', 'N/A')} ({column.get('Data Type', 'N/A')})")
        
        # Save test result (use relative paths)
        test_path = f"Files/lineage/test/semantic_model_{TEST_MODEL_ID}.json"
        mssparkutils.fs.mkdirs("Files/lineage/test")
        mssparkutils.fs.put(test_path, json.dumps(test_result, indent=2), overwrite=True)
        print(f"\n💾 Test result saved to: {test_path}")
        
    else:
        print(f"\n❌ FAILED: {test_result['metadata']['errorMessage']}")
        
else:
    print("⚠️  Update TEST_WORKSPACE_ID and TEST_MODEL_ID to run test")
    print("\nHow to get IDs:")
    print("1. Workspace ID: From workspace URL (last GUID)")
    print("   Example: https://app.fabric.microsoft.com/groups/{workspace-id}/list")
    print("\n2. Model ID: Open Semantic Model → Settings → Copy object ID")

## 🧪 Verify Saved Data

In [ ]:
# List all extracted files
try:
    files = mssparkutils.fs.ls("Files/lineage/raw/semantic_models")
    print(f"📁 Found {len(files)} extracted file(s):\n")
    
    for file in files:
        if file.name.endswith('.json'):
            # Read and display summary
            content = mssparkutils.fs.head(f"Files/lineage/raw/semantic_models/{file.name}", 10000)
            data = json.loads(content)
            
            print(f"   {file.name}")
            print(f"      Timestamp: {data.get('timestamp', 'N/A')}")
            print(f"      Status: {data.get('metadata', {}).get('status', 'N/A')}")
            if 'counts' in data.get('data', {}):
                counts = data['data']['counts']
                print(f"      Metrics: {counts['measures']}M, {counts['columns']}C, {counts['relationships']}R")
            print()
            
except Exception as e:
    print(f"❌ No files found or error reading: {e}")
    print("   Run extraction first (cells 4-8)")

---
## 📋 Troubleshooting

### Common Issues:

**1. "ModuleNotFoundError: No module named 'sempy'"**
- Attach Fabric Environment with semantic-link packages (top toolbar → Environment dropdown)
- Verify environment is published (not Draft status)
- Run `00_setup.ipynb` to verify environment configuration

**2. "401 Unauthorized" or "403 Forbidden"**
- Verify workspace permissions (Contributor or higher required)
- Check model permissions (viewer access minimum)
- Ensure correct workspace/model IDs

**3. "Empty DataFrames"**
- Model might have no measures/columns (unlikely)
- Check if semantic-link-labs supports the model version
- Try different model

**4. "Timeout" or "Performance Issues"**
- Large models (1000+ measures) take 30-60 seconds
- Consider extracting fewer workspaces at once
- Use cell 11 to test single model first

### Expected Performance:
- Small model (< 50 measures): 5-10 seconds
- Medium model (50-500 measures): 10-30 seconds
- Large model (> 500 measures): 30-60 seconds

### Next Steps After Testing:
1. ✅ Verify extraction works with test model
2. Update `WORKSPACE_IDS` in cell 4 with target workspaces
3. Run full extraction (cells 4-8)
4. Check lakehouse for results
5. Proceed to `02_extract_reports.ipynb`